In [ ]:
import torch
from transformers import PreTrainedTokenizerFast
from transformers import BartForConditionalGeneration

tokenizer = PreTrainedTokenizerFast.from_pretrained('gogamza/kobart-summarization')
model = BartForConditionalGeneration.from_pretrained('gogamza/kobart-summarization')

text = """행정안전부가 내년 공공 정보시스템의 민간 클라우드 이전과 재해복구(DR), 노후 전산장비 교체 등에 대규모 예산을 투입한다. 인공지능(AI) 행정 확대에 맞춰 공공 IT 인프라의 안정성과 연속성을 강화하고 범정부 정보시스템의 클라우드 전환에도 속도를 낸다는 목표다.
행안부는 공공 정보시스템 민간 클라우드 이전에 548억원을 신규 편성하고 범정부 이중운영시스템 구축에 2716억원을 투입하는 내용을 담은 2027년도 예산안을 2일 발표했다.
행안부는 올 초 국가AI전략위원회가 발표한 'AI정부 인프라 거버넌스 혁신 추진방향'에 발맞춰 정부 데이터센터에 집중된 시스템을 민간 클라우드 등에 분산하는 방안을 추진하고 있다. 이에 내년 공공 정보시스템의 민간 클라우드 이전을 위한 예산으로 548억원을 새로 배정했다."""

raw_input_ids = tokenizer.encode(text)
input_ids = [tokenizer.bos_token_id] + raw_input_ids + [tokenizer.eos_token_id]

summary_ids = model.generate(torch.tensor([input_ids]), min_length=35, num_beams=5)

print("요약문:" ,tokenizer.decode(summary_ids.squeeze().tolist(), skip_specialtokens=True))

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 13314.44it/s]
c:\Users\IAM87\Documents\ax_study\04_MACHINE_LEARNING\.venv\Lib\site-packages\transformers\generation\utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
c:\Users\IAM87\Documents\ax_study\04_MACHINE_LEARNING\.venv\Lib\site-packages\transformers\generation\utils.py:1700: UserWarning: Unfeasible length constraints: `min_length` (35) is larger than the maximum possible length (21). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length. Note that `max_length` is set to 21, its default value.
  warnings.warn(


요약문: <s> 행정안전부가 인공지능(AI) 행정 확대에 맞춰 공공 IT 인프라의 안정성과 연속성을 강화하고 범정부


In [14]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import pandas as pd

tokenizer = AutoTokenizer.from_pretrained("monologg/kobert", trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained("rkdaldus/ko-sent5-classification")

text = "이번 시험을 위해 밤을 새우며 정말 열심히 준비했는데 생각보다 점수가 너무 낮게 나와서 속상하고 눈물이 나네."
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

probabilities = F.softmax(logits, dim=1)

emotion_labels = ['분노','불안','기쁨','평온','슬픔']

df = pd.DataFrame({
    "감정": emotion_labels,
    "확률": probabilities[0].tolist()
    })

df


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12788.88it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: rkdaldus/ko-sent5-classification
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,감정,확률
0,분노,0.324884
1,불안,0.108867
2,기쁨,0.128600
3,평온,0.148265
4,슬픔,0.289384


In [25]:
from transformers import pipeline
import torch

gemma_identifier = "google/gemma-2b-it"

gemma_generator = pipeline(
    "text-generation",
    model = gemma_identifier,
    dtype=torch.bfloat16,
    device_map="auto"
)

user_diaglogue = [{"role": "user", "content": "클라우드 컴퓨팅의 개념을 딱 한 문장으로 재미있게 비유해서 알려줘"}]

outputs = gemma_generator(user_diaglogue,max_new_tokens=200)
outputs

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2b-it.
401 Client Error. (Request ID: Root=1-6a97d667-6bad72625d5fca430a041e52;1d085511-2b1b-4c5c-ab0a-7a8db7d66a6e)

Cannot access gated repo for url https://huggingface.co/google/gemma-2b-it/resolve/main/config.json.
Access to model google/gemma-2b-it is restricted. You must have access to it and be authenticated to access it. Please log in.